## OpenVLA简介

目前很多无人机的VLN/VLA的研究，使用的基础模型都是OpenVLA. OpenVLA以其简单、可扩展性强等特点，成为VLA模型的典型代表。

OpenVLA（Open Vision-Language-Action）是由斯坦福大学、UC Berkeley 和 NVIDIA 等机构联合推出的一个开源视觉-语言-动作（Vision-Language-Action, VLA）基础模型，旨在为机器人提供通用的感知、理解和执行能力。它代表了具身智能（Embodied AI）领域的重要进展，目标是构建能够理解人类自然语言指令、感知环境并通过动作与之交互的通用机器人智能体。

### 核心特点

1. **多模态融合**：
   - OpenVLA 能够同时处理视觉（图像）、语言（文本指令）和动作（机器人控制信号）三种模态。
   - 通过统一的架构，模型可以根据自然语言指令和当前视觉输入，预测出合适的机器人动作（如关节角度、末端执行器位姿等）。

2. **大规模训练数据**：
   - 基于超过 **100 万条**机器人操作轨迹（来自多个现有机器人数据集，如 Bridge、RT-X、Franka Kitchen 等）进行训练。
   - 数据涵盖多种任务、场景、机器人类型，增强了模型的泛化能力。

3. **开源与可扩展性**：
   - OpenVLA 是完全开源的（包括模型权重、训练代码、推理工具等），托管在 GitHub 上。
   - 支持微调（fine-tuning）以适应特定机器人平台或任务。
   - 提供 Hugging Face 集成，便于社区使用和部署。

4. **基于 Prismatic VLM 架构**：
   - OpenVLA 基于 Prismatic Vision-Language Models（一种高效、可扩展的 VLM 架构），并扩展了动作预测头（action head）。
   - 支持多种动作空间（如连续控制、离散动作等）。

5. **高性能与通用性**：
   - 在多个机器人基准测试中表现优异，甚至在未见过的任务或环境中展现出零样本（zero-shot）或少样本（few-shot）迁移能力。
   - 可部署在真实机器人上，实现端到端的指令执行。

### 应用场景

- 家庭服务机器人（如“把杯子放到水槽里”）
- 工业自动化（如按指令分拣物品）
- 教育与研究平台（用于具身智能算法开发）
- 人机协作（通过自然语言与机器人交互）

### 项目资源

- GitHub 仓库：https://github.com/openvla/openvla  
- 官网/文档：https://openvla.github.io  
- Hugging Face 模型库：https://huggingface.co/openvla  

### 意义

OpenVLA 的发布标志着机器人基础模型（Robot Foundation Models）进入开源时代，降低了具身智能研究的门槛，推动了从“专用机器人”向“通用智能体”的演进。


而应用到无人机上，只需要使用无人机相关的视觉和指令数据，以及对应的动作数据，进行微调即可。


In [5]:
# Install minimal dependencies (`torch`, `transformers`, `timm`, `tokenizers`, ...)
# > pip install -r https://raw.githubusercontent.com/openvla/openvla/main/requirements-min.txt
from transformers import AutoModelForVision2Seq, AutoProcessor
from PIL import Image
import torch

In [6]:
# Load Processor & VLA
processor = AutoProcessor.from_pretrained("openvla/openvla-7b", trust_remote_code=True)
vla = AutoModelForVision2Seq.from_pretrained(
    "openvla/openvla-7b",
    attn_implementation="flash_attention_2",  # [Optional] Requires `flash_attn`
    torch_dtype=torch.bfloat16, 
    low_cpu_mem_usage=True, 
    trust_remote_code=True
).to("cuda:0")

'(ProtocolError('Connection aborted.', ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None)), '(Request ID: 0f48046b-c2d0-49d5-9cac-5ca08ff01a6e)')' thrown while requesting HEAD https://huggingface.co/openvla/openvla-7b/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None)), '(Request ID: b591d05b-f3b4-4a61-bdfa-85a933b31388)')' thrown while requesting HEAD https://huggingface.co/openvla/openvla-7b/resolve/main/processor_config.json
Retrying in 2s [Retry 2/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None)), '(Request ID: 903a61aa-ddb1-417d-9c24-5e65d261c3d3)')' thrown while requesting HEAD https://huggingface.co/openvla/openvla-7b/resolve/main/processor_config.json
Retrying in 4s [Retry 3/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None)

ImportError: FlashAttention2 has been toggled on, but it cannot be used due to the following error: the package flash_attn seems to be not installed. Please refer to the documentation of https://huggingface.co/docs/transformers/perf_infer_gpu_one#flashattention-2 to install Flash Attention 2.

In [4]:
from PIL import Image
import torch

# 1. 读取本地图片（替换原来的 get_from_camera）
image_path = "img/cup.png"  # 👈 改成你的图片路径
image = Image.open(image_path).convert("RGB")  # 确保是 RGB 格式

# 2. 构造指令提示（例如让机器人拿起杯子）
instruction = "pick up the cup from the table"
prompt = f"In: What action should the robot take to {instruction}?\nOut:"

# 3. 使用 processor 处理 prompt 和 image
inputs = processor(prompt, image).to("cuda:0", dtype=torch.bfloat16)

# 4. 预测动作（7-DoF，使用 BridgeV2 的反归一化参数）
action = vla.predict_action(**inputs, unnorm_key="bridge_orig", do_sample=False)

print("Predicted action:", action)

NameError: name 'vla' is not defined

In [2]:
!pip install timm

   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.5 MB 5.6 MB/s eta 0:00:01
   ------------------------------------- -- 2.4/2.5 MB 9.6 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 5.6 MB/s  0:00:00
